# Лекция 4. Архитектура глубоких сетей — код

Код из лекции: почему глубокая сеть не учится, инициализация Кайминга, остаточные соединения, BatchNorm и LayerNorm, pre-norm и post-norm, bottleneck, ворота, DropPath и шаблон глубокой сети.

Ячейки выполняются **по порядку**: каждая следующая использует переменные из предыдущих. Проще всего — «Среда выполнения → Выполнить все».

В Colab всё нужное уже установлено; на своём компьютере — `pip install torch scikit-learn`. Сети обучаются на процессоре; самые долгие ячейки (сети из 100 слоёв) работают по нескольку минут. Последние знаки чисел на вашей машине могут немного отличаться.

## Глава 1. Почему глубокую сеть трудно обучить

### 1.2. Эксперимент: добавляем слои

Датасет рукописных цифр: 1797 картинок 8×8, то есть 64 признака, и 10 классов. Функция `fit` обучает модель 30 эпох и возвращает потерю на обучении и точность на валидации.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# X — 1797 картинок по 64 числа (яркости пикселей 8×8), y — цифра на картинке (от 0 до 9)
X, y = load_digits(return_X_y=True)

# 75% примеров — для обучения, 25% — для валидации;
# stratify=y: в обеих частях цифры встречаются в одинаковых пропорциях
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# стандартизация, как в лекции 2: среднее и разброс считаем только по обучающим данным
scaler = StandardScaler().fit(X_train)
X_train = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
X_val = torch.tensor(scaler.transform(X_val), dtype=torch.float32)
y_train, y_val = torch.tensor(y_train), torch.tensor(y_val)   # номера классов — целые числа
print(X_train.shape, X_val.shape)

def fit(model, epochs=30, lr=1e-3):
    """Обучает модель и возвращает потерю на обучении и точность на валидации."""
    torch.manual_seed(0)                              # одинаковое перемешивание для всех моделей
    # DataLoader выдаёт обучающие данные батчами по 64 примера, каждую эпоху в новом порядке
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)   # Adam — оптимизатор из лекции 3
    loss_fn = nn.CrossEntropyLoss()                   # многоклассовая классификация (лекция 3)
    for epoch in range(epochs):
        model.train()                                 # режим обучения: dropout и BatchNorm включены
        for xb, yb in loader:                         # пять шагов цикла обучения из лекции 2:
            optimizer.zero_grad()                     # 1. обнулить старые градиенты
            loss = loss_fn(model(xb), yb)             # 2–3. прямой проход и потеря
            loss.backward()                           # 4. обратный проход: градиенты
            optimizer.step()                          # 5. шаг оптимизатора
    model.eval()                                      # режим проверки
    with torch.no_grad():                             # при проверке градиенты не нужны
        train_loss = loss_fn(model(X_train), y_train).item()
        # argmax — номер класса с наибольшим логитом, то есть ответ модели
        val_acc = (model(X_val).argmax(dim=1) == y_val).float().mean().item()
    return train_loss, val_acc

def count_linear(model):
    """Сколько линейных слоёв в модели."""
    # model.modules() перебирает все части модели, включая вложенные блоки
    return sum(isinstance(m, nn.Linear) for m in model.modules())

**Что получилось.** 1347 картинок для обучения и 450 для валидации, по 64 признака.

Простые сети разной глубины: `Linear → ReLU`, повторённые много раз, ширина 64. Обучаем каждую одинаково.

In [ ]:
def plain_mlp(depth, width=64):
    """Простая сеть: depth скрытых слоёв Linear → ReLU и выходной слой на 10 классов."""
    layers = [nn.Linear(64, width), nn.ReLU()]        # первый слой: 64 пикселя → width нейронов
    for _ in range(depth - 1):                        # остальные скрытые слои: width → width
        layers += [nn.Linear(width, width), nn.ReLU()]
    layers.append(nn.Linear(width, 10))               # выходной слой: 10 логитов, по одному на цифру
    return nn.Sequential(*layers)                     # * распаковывает список в аргументы

for depth in [2, 10, 30, 100]:
    torch.manual_seed(0)                              # одинаковые начальные веса при каждом запуске
    model = plain_mlp(depth)
    loss, acc = fit(model)
    print(f"линейных слоёв: {count_linear(model):3d} | потеря на обучении {loss:.4f} | точность на валидации {acc:.3f}")

* 3 и 11 слоёв — сеть учится: точность 97.1% и 93.8%.
* 31 и 101 слой — потеря 2.3027, точность 9.8%: **сеть не выучила ничего**. $2.3027 = \ln 10$ — потеря модели, которая всем 10 цифрам даёт вероятность 10%.

Это не переобучение: сеть не может выучить даже обучающие данные. Такую проблему называют **деградацией**.

### 1.3. Что сломалось: сигнал не доходит

Заглянем внутрь необученной сети из 31 слоя: выходы для 5 разных картинок, разброс между примерами после слоёв 1, 10, 20, 30 и размер градиента весов слоёв 1, 10, 20, 31.

In [ ]:
torch.manual_seed(0)
model = plain_mlp(30)                                 # необученная сеть из 31 линейного слоя

with torch.no_grad():
    print(model(X_train[:5])[:, :4])                  # выходы для 5 разных картинок (первые 4 логита)

# Насколько выходы ReLU различаются между примерами, после слоёв 1, 10, 20, 30.
# Пропускаем 256 картинок через сеть слой за слоем и после каждой ReLU
# считаем разброс (стандартное отклонение) каждого нейрона по примерам.
h, spread = X_train[:256], []
with torch.no_grad():
    for layer in model:                               # nn.Sequential можно перебирать по слоям
        h = layer(h)
        if isinstance(layer, nn.ReLU):
            # std(dim=0) — разброс по примерам для каждого нейрона, mean() — среднее по нейронам
            spread.append(h.std(dim=0).mean().item())
print("разброс между примерами:", [f"{spread[i]:.0e}" for i in [0, 9, 19, 29]])

# Размер градиента у весов слоёв 1, 10, 20, 31: один обратный проход без обучения
nn.CrossEntropyLoss()(model(X_train[:256]), y_train[:256]).backward()
linears = [m for m in model if isinstance(m, nn.Linear)]
# norm() — длина градиента: корень из суммы квадратов всех его чисел
print("градиент весов:      ", [f"{linears[i].weight.grad.norm().item():.0e}" for i in [0, 9, 19, 30]])

* Выходы для пяти разных картинок совпадают: сеть «не видит» входа.
* Разброс между примерами тает: 0.3 → $8 \cdot 10^{-5}$ → $10^{-8}$ — примерно в 2.5 раза за слой.
* Градиент первого слоя — $5 \cdot 10^{-12}$, последнего — $4 \cdot 10^{-2}$: первые слои почти не учатся.

### 1.4. Может, просто правильно подобрать веса?

Инициализация Кайминга: дисперсия весов $2 / n_{\text{in}}$. Те же сети, только другие начальные веса.

In [ ]:
def kaiming_init(model):
    """Заменяет начальные веса всех линейных слоёв на инициализацию Кайминга."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            # случайные веса с дисперсией 2 / n_in, где n_in — число входов нейрона
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            nn.init.zeros_(m.bias)                    # смещения — нули

for depth in [30, 100]:
    torch.manual_seed(0)
    model = plain_mlp(depth)
    kaiming_init(model)                               # та же сеть, только другие начальные веса
    loss, acc = fit(model)
    print(f"линейных слоёв: {count_linear(model):3d} | потеря на обучении {loss:.4f} | точность на валидации {acc:.3f}")

**Что получилось.** 31 слой — 92.2%: помогло. 101 слой — 33%: снова провал. Формула Кайминга верна только в среднем, а на 100 слоях отклонения перемножаются; к тому же при обучении баланс множителей никто не поддерживает.

## Глава 2. Остаточные соединения

### 2.2. Почему в остаточной сети градиент не затухает

Остаточный блок $y = x + F(x)$ и сеть из 15 таких блоков — 32 линейных слоя. Смотрим на неё до обучения.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        # ветка F: два линейных слоя с ReLU между ними; ширина на входе и выходе одинаковая
        self.f = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))

    def forward(self, x):
        return x + self.f(x)                          # вход + поправка: y = x + F(x)

def residual_mlp(n_blocks, width=64):
    """Входной слой, n_blocks остаточных блоков и выходной слой."""
    return nn.Sequential(nn.Linear(64, width),                          # 64 пикселя → width
                         *[ResidualBlock(width) for _ in range(n_blocks)],  # n_blocks одинаковых блоков
                         nn.Linear(width, 10))                          # width → 10 логитов

torch.manual_seed(0)
# 15 блоков по 2 слоя + входной и выходной = 32 линейных слоя
model = residual_mlp(15)
with torch.no_grad():
    print(model(X_train[:5])[:, :4])                  # те же 5 картинок, что в разделе 1.3

# градиенты слоёв 1, 11, 21, 32 — так же, как для простой сети
nn.CrossEntropyLoss()(model(X_train[:256]), y_train[:256]).backward()
# слои спрятаны внутри блоков, поэтому ищем их через modules(), а не перебором model
linears = [m for m in model.modules() if isinstance(m, nn.Linear)]
print("градиент весов:", [f"{linears[i].weight.grad.norm().item():.0e}" for i in [0, 10, 20, 31]])

**Что получилось.** Выходы для пяти картинок теперь **разные**, а градиенты на всех слоях одного порядка — от 0.2 до 1. В простой сети они различались в десять миллиардов раз.

### 2.3. Обучаем остаточные сети из 32 и 102 слоёв

Обучаем остаточные сети из 32 и 102 слоёв — с инициализацией по умолчанию.

In [ ]:
for n_blocks in [15, 50]:                             # около 30 и около 100 линейных слоёв
    torch.manual_seed(0)
    model = residual_mlp(n_blocks)                    # инициализация по умолчанию, без Кайминга
    loss, acc = fit(model)
    print(f"линейных слоёв: {count_linear(model):3d} | потеря на обучении {loss:.4f} | точность на валидации {acc:.3f}")

**Что получилось.** Обе сети обучились: 97.6% и 98.0%. Изменили только `x + ...` в `forward`, инициализация — по умолчанию.

| Глубина | Простая сеть | Простая + Кайминг | Остаточная сеть |
| --- | --- | --- | --- |
| около 30 слоёв | 9.8% | 92.2% | **97.6%** |
| около 100 слоёв | 9.8% | 33.3% | **98.0%** |

### 2.4. Если размеры не совпадают: проекция

Если блок меняет ширину (64 → 128), в обходной путь ставят **проекцию** — линейный слой.

In [ ]:
class ResidualBlockProj(nn.Module):
    """Остаточный блок, который меняет ширину: d_in → d_out."""
    def __init__(self, d_in, d_out):
        super().__init__()
        # ветка F сразу переводит d_in признаков в d_out
        self.f = nn.Sequential(nn.Linear(d_in, d_out), nn.ReLU(), nn.Linear(d_out, d_out))
        # проекция: приводит x к ширине d_out; смещение не нужно — его уже есть в ветке
        self.proj = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        return self.proj(x) + self.f(x)               # y = P(x) + F(x)

block = ResidualBlockProj(64, 128)
x = torch.randn(32, 64)                               # батч: 32 примера по 64 признака
# формы выхода ветки F(x), проекции P(x) и всего блока: все три должны быть (32, 128)
print(block.f(x).shape, block.proj(x).shape, block(x).shape)

**Что получилось.** И ветка $F(x)$, и проекция $P(x)$ выдают по 128 признаков на каждый из 32 примеров, поэтому их можно сложить.

## Глава 3. Нормализация

### 3.2. BatchNorm: нормализация по батчу

BatchNorm вручную и в PyTorch. Батч из 4 примеров, два признака очень разного масштаба.

In [ ]:
x = torch.tensor([[1.0, 100.0],
                  [2.0, 200.0],
                  [3.0, 300.0],
                  [4.0, 400.0]])                      # батч: 4 примера (строки), 2 признака (столбцы)

# dim=0 — считаем вдоль строк, то есть по примерам: отдельно для каждого признака
mean = x.mean(dim=0)                                  # среднее каждого признака по батчу
var = x.var(dim=0, unbiased=False)                    # дисперсия каждого признака по батчу
print("среднее:", mean, " дисперсия:", var)
print((x - mean) / torch.sqrt(var + 1e-5))            # нормализация вручную; 1e-5 — это ε

bn = nn.BatchNorm1d(2)                                # 2 — число признаков
print(bn(x))                                          # то же самое (модель в режиме train)

**Что получилось.** Оба признака превратились в одни и те же числа $-1.34, -0.45, 0.45, 1.34$: среднее 0, разброс 1. Масштаб исчез, осталось только положение примера относительно остальных. `nn.BatchNorm1d` выдал то же самое.

### 3.3. BatchNorm при обучении и при проверке

BatchNorm в режимах `train` и `eval`: копим статистику на 100 «обучающих» батчах, потом нормализуем один пример в каждом режиме.

In [ ]:
bn = nn.BatchNorm1d(2)                                # новый слой: накопленной статистики пока нет
# «Обучение»: 100 случайных батчей по 64 примера.
# У первого признака среднее 2.5 и разброс 1, у второго — среднее 250 и разброс 100.
# Слой в режиме train: каждый вызов обновляет накопленные среднее и дисперсию.
for _ in range(100):
    # randn даёт среднее 0 и разброс 1; умножение задаёт разброс, сложение — среднее
    bn(torch.randn(64, 2) * torch.tensor([1.0, 100.0]) + torch.tensor([2.5, 250.0]))
print("накопленное среднее:", bn.running_mean)        # BatchNorm запомнил статистику данных
print("накопленная дисперсия:", bn.running_var)

bn.eval()                                             # режим проверки
# один пример: нормализуется накопленной статистикой
print(bn(torch.tensor([[2.5, 250.0]])))

bn.train()                                            # обратно в режим обучения
try:
    bn(torch.tensor([[2.5, 250.0]]))                  # один пример в режиме train
except ValueError as e:                               # ловим ошибку, чтобы показать её текст
    print("ошибка:", e)

* Накопленная статистика — около $(2.5, 250)$ и $(1, 10^4)$: BatchNorm запомнил данные.
* В режиме `eval` пример $(2.5, 250)$ стал почти $(0, 0)$ — он нормализован накопленной статистикой.
* В режиме `train` один пример вызвал ошибку: дисперсию по батчу из одного примера не посчитать.

Перед проверкой модели — `model.eval()`, перед продолжением обучения — `model.train()`.

### 3.4. LayerNorm: нормализация по признакам

LayerNorm считает среднее и дисперсию по признакам каждого примера отдельно.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                  [10.0, 20.0, 30.0, 40.0]])          # 2 примера (строки), 4 признака (столбцы)

# dim=1 — считаем вдоль столбцов, то есть по признакам: отдельно для каждого примера;
# keepdim=True оставляет форму (2, 1), чтобы вычитание сработало для каждой строки
mean = x.mean(dim=1, keepdim=True)                    # среднее каждого примера по его признакам
var = x.var(dim=1, unbiased=False, keepdim=True)      # дисперсия каждого примера
print("среднее:", mean.squeeze(), " дисперсия:", var.squeeze())   # squeeze убирает лишнюю ось
print((x - mean) / torch.sqrt(var + 1e-5))            # нормализация вручную

ln = nn.LayerNorm(4)                                  # 4 — число признаков
print(ln(x))                                          # то же самое
print(ln(x[:1]))                                      # один пример — работает так же

**Что получилось.** Каждый пример нормализован своими средним и дисперсией; после нормализации оба стали одинаковыми. Один пример LayerNorm обрабатывает без проблем — поэтому он одинаково работает при обучении и при проверке.

### 3.5. Одной нормализации мало

Хватит ли одной нормализации? Простая сеть из 31 слоя с BatchNorm после каждого линейного слоя, без остаточных соединений.

In [ ]:
def plain_bn(depth, width=64):
    """Простая сеть, где после каждого линейного слоя стоит BatchNorm."""
    # первый блок: 64 признака на входе (пиксели 8×8) → width нейронов
    layers = [nn.Linear(64, width), nn.BatchNorm1d(width), nn.ReLU()]
    # ещё depth − 1 одинаковых блоков Linear → BatchNorm → ReLU;
    # обходных путей нет: сигнал проходит через каждый слой по очереди
    for _ in range(depth - 1):
        layers += [nn.Linear(width, width), nn.BatchNorm1d(width), nn.ReLU()]
    layers.append(nn.Linear(width, 10))               # выходной слой: 10 логитов, по одному на цифру
    return nn.Sequential(*layers)                     # обычная цепочка слоёв, без x + F(x)

torch.manual_seed(0)
model = plain_bn(30)                                  # 30 блоков + выходной слой = 31 линейный слой, как в разделе 1.3
# градиенты слоёв 1, 10, 20, 31 до обучения
nn.CrossEntropyLoss()(model(X_train[:256]), y_train[:256]).backward()
linears = [m for m in model if isinstance(m, nn.Linear)]   # только линейные слои, без BatchNorm и ReLU
print("градиент весов:", [f"{linears[i].weight.grad.norm().item():.0e}" for i in [0, 9, 19, 30]])

loss, acc = fit(model)                                # обучаем, как все сети в лекции
print(f"потеря на обучении {loss:.4f} | точность на валидации {acc:.3f}")

**Что получилось.** Сеть снова не обучилась: потеря 2.30, почти $\ln 10$. Градиенты теперь не затухают, а **растут** к входу: у первого слоя 90, у последнего 0.4. Главную работу делает остаточное соединение, нормализация — его помощник.

### 3.6. Куда ставить нормализацию: pre-norm и post-norm

Куда ставить нормализацию: pre-norm $x + F(\operatorname{Norm}(x))$ и post-norm $\operatorname{Norm}(x + F(x))$. Сравним на сетях из 50 блоков (102 линейных слоя).

In [ ]:
class PreNormBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)               # слой нормализации
        # ветка F — такая же, как в остаточном блоке из главы 2
        self.f = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))

    def forward(self, x):
        return x + self.f(self.norm(x))               # нормализуем вход ветки, обходной путь чистый

class PostNormBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)               # те же слои, что в PreNormBlock
        self.f = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))

    def forward(self, x):
        return self.norm(x + self.f(x))               # нормализуем сумму: x проходит через norm

# одинаковые сети из 50 блоков, отличаются только местом нормализации
for name, Block in [("pre-norm ", PreNormBlock), ("post-norm", PostNormBlock)]:
    torch.manual_seed(0)
    # входной слой 64 → 64, 50 блоков, выходной слой на 10 логитов
    model = nn.Sequential(nn.Linear(64, 64), *[Block(64) for _ in range(50)], nn.Linear(64, 10))
    loss, acc = fit(model)
    print(f"{name}, {count_linear(model)} слоёв | потеря на обучении {loss:.4f} | точность на валидации {acc:.3f}")

**Что получилось.** Обе сети обучились, но pre-norm лучше: потеря на обучении 0.0001 против 0.068, точность 97.6% против 95.8%. В pre-norm обходной путь чистый — $x$ без изменений доходит от входа до выхода.

## Глава 4. Что ещё кладут внутрь блока

### 4.1. Bottleneck и inverted bottleneck

Bottleneck и inverted bottleneck: сколько параметров у трёх вариантов ветки при $d = 256$.

In [ ]:
def n_params(module):
    """Сколько обучаемых чисел (весов и смещений) в модуле."""
    return sum(p.numel() for p in module.parameters())    # numel — число элементов тензора

d = 256                                               # ширина входа и выхода ветки
# три варианта ветки F; отличаются только шириной среднего слоя
plain_branch = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, d))              # d → d → d
bottleneck = nn.Sequential(nn.Linear(d, d // 4), nn.ReLU(), nn.Linear(d // 4, d))      # d → d/4 → d
inverted = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d))          # d → 4d → d
for name, m in [("d → d → d    ", plain_branch), ("d → d/4 → d  ", bottleneck), ("d → 4d → d   ", inverted)]:
    print(f"{name} параметров: {n_params(m):7d}")

**Что получилось.** Bottleneck — в 4 раза меньше параметров (33 088), inverted bottleneck — в 4 раза больше (525 568). Inverted bottleneck с расширением в 4 раза — ветка FFN в трансформерах.

### 4.2. Ворота: пропустить или не пропустить

Ворота GLU: значения и ворота вычисляют два разных слоя, выход — их поэлементное произведение.

In [ ]:
x = torch.tensor([[1.0, -2.0, 3.0]])                  # один пример с 3 признаками
value = nn.Linear(3, 4)                               # что передать дальше: 4 значения
gate = nn.Linear(3, 4)                                # ворота для каждого из 4 значений
# для наглядности заменяем веса по умолчанию своими:
# normal_ берёт веса из нормального распределения со средним 0 и разбросом 1 —
# это крупнее, чем по умолчанию (у Linear(3, 4) разброс весов около 0.33);
# с крупными весами ворота заметно расходятся к 0 или к 1
torch.manual_seed(0)
nn.init.normal_(value.weight); nn.init.normal_(gate.weight)
nn.init.zeros_(value.bias); nn.init.zeros_(gate.bias)  # смещения — нули

v = value(x)
g = torch.sigmoid(gate(x))                            # сигмоида переводит в числа от 0 до 1
print("значения:", v)
print("ворота:  ", g)
print("выход:   ", g * v)                             # поэлементное произведение: GLU(x) = g ⊙ v

**Что получилось.** У первого признака ворота почти закрыты (0.002), и на выходе почти ноль. У остальных ворота открыты на 87–96%. Значение второго признака $-1.46$ прошло: ReLU обнулила бы его, а ворота пропустили.

## Глава 5. Регуляризация и сборка глубокой сети

### 5.2. DropPath: выключаем блок целиком

DropPath: при обучении для каждого примера ветка блока выключается целиком с вероятностью $p$.

In [ ]:
def drop_path(branch_out, p, training):
    """Для каждого примера с вероятностью p обнуляет выход ветки целиком."""
    if not training or p == 0:                        # при проверке ветка работает всегда
        return branch_out
    # для каждого примера (строки) — одно случайное число; сравнение с p даёт True или False,
    # .float() превращает их в 1.0 (оставить ветку) и 0.0 (выключить)
    keep = (torch.rand(branch_out.shape[0], 1) > p).float()
    # форма keep — (батч, 1): broadcasting (лекция 2) умножает на него всю строку
    return branch_out * keep / (1 - p)                # деление сохраняет среднее

torch.manual_seed(0)
f_x = torch.ones(6, 3)                                # выход ветки для батча из 6 примеров (все единицы)
print(drop_path(f_x, p=0.5, training=True))           # обучение
print(drop_path(f_x, p=0.5, training=False))          # проверка

**Что получилось.** При обучении у 4 примеров из 6 ветка выключена (нулевые строки), у остальных она увеличена в 2 раза — поделена на $1 - p$, чтобы в среднем вклад не изменился. При проверке ветка работает всегда.

### 5.3. Собираем глубокую сеть

Собираем шаблон глубокой сети: pre-norm, расширение в 4 раза, GELU, dropout, DropPath, финальная нормализация.

In [ ]:
class Block(nn.Module):
    """Pre-norm остаточный блок с расширением, GELU, dropout и DropPath."""
    def __init__(self, width, expansion=4, dropout=0.1, drop_path_p=0.0):
        super().__init__()
        self.norm = nn.LayerNorm(width)               # pre-norm: нормализуем вход ветки
        self.f = nn.Sequential(                       # ветка F — inverted bottleneck
            nn.Linear(width, expansion * width),      # расширяем: width → 4·width
            nn.GELU(),                                # нелинейность
            nn.Dropout(dropout),                      # dropout только внутри ветки
            nn.Linear(expansion * width, width),      # сжимаем обратно: 4·width → width
        )
        self.drop_path_p = drop_path_p                # вероятность выключить ветку целиком

    def forward(self, x):
        # self.training — True после model.train() и False после model.eval()
        return x + drop_path(self.f(self.norm(x)), self.drop_path_p, self.training)

class DeepMLP(nn.Module):
    """Глубокая сеть: stem, n_blocks блоков Block, финальная нормализация и head."""
    def __init__(self, d_in, n_classes, width=64, n_blocks=12, max_drop_path=0.1):
        super().__init__()
        self.stem = nn.Linear(d_in, width)            # stem: вход → рабочая ширина
        self.blocks = nn.Sequential(*[
            # вероятность DropPath растёт линейно: от max_drop_path / n_blocks до max_drop_path
            Block(width, drop_path_p=max_drop_path * (i + 1) / n_blocks)
            for i in range(n_blocks)
        ])
        self.norm = nn.LayerNorm(width)               # финальная нормализация перед головой
        self.head = nn.Linear(width, n_classes)       # head: рабочая ширина → логиты

    def forward(self, x):
        # stem → блоки → финальная нормализация → head
        return self.head(self.norm(self.blocks(self.stem(x))))

torch.manual_seed(0)
model = DeepMLP(64, 10)                               # 64 пикселя на входе, 10 цифр на выходе
print("линейных слоёв:", count_linear(model), "| параметров:", n_params(model))
# вероятности DropPath: первые три блока и последний
print("p DropPath по блокам:", [round(b.drop_path_p, 3) for b in model.blocks][:3], "...", round(model.blocks[-1].drop_path_p, 3))

**Что получилось.** 12 блоков по 2 линейных слоя, плюс stem и head — 26 линейных слоёв и около 400 тысяч параметров. Вероятность DropPath растёт от 0.008 в первом блоке до 0.1 в последнем.

### 5.4. Проверки перед обучением

Проверки перед обучением (санити-чеки): формы, потеря до обучения, градиенты, способность выучить маленький батч.

In [ ]:
torch.manual_seed(0)
model = DeepMLP(64, 10)
xb, yb = X_train[:16], y_train[:16]                   # маленький батч для проверок

# 1. Формы: на выходе должно быть по 10 логитов на каждый из 16 примеров
print("1. выход:", model(xb).shape)

# 2. Потеря до обучения: должна быть близка к ln 10 (лекция 2, раздел 3.2)
model.eval()
with torch.no_grad():
    print(f"2. потеря до обучения: {nn.CrossEntropyLoss()(model(X_train), y_train).item():.3f}  (ln 10 = 2.303)")

# 3. Градиенты доходят до первого слоя: сравниваем первый (stem) и последний (head)
model.train()
model.zero_grad()                                     # обнуляем градиенты модели перед backward
nn.CrossEntropyLoss()(model(xb), yb).backward()
print(f"3. градиент: первый слой {model.stem.weight.grad.norm():.0e}, последний {model.head.weight.grad.norm():.0e}")

# 4. Переобучение на маленьком батче: 200 шагов на одних и тех же 16 примерах,
#    исправная сеть должна «зазубрить» их почти до нулевой потери
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(200):                               # тот же цикл обучения, что в fit, но на одном батче
    opt.zero_grad()                                   # обнулить старые градиенты
    loss = nn.CrossEntropyLoss()(model(xb), yb)       # прямой проход и потеря
    loss.backward()                                   # обратный проход
    opt.step()                                        # шаг оптимизатора
print(f"4. потеря на 16 примерах после 200 шагов: {loss.item():.4f}")

1. Выход формы `(16, 10)` — по 10 логитов на пример.
2. Потеря до обучения 2.445 — близко к $\ln 10 = 2.303$.
3. Градиенты первого и последнего слоя одного порядка — сигнал доходит.
4. 16 примеров за 200 шагов выучены почти до нулевой потери.

Если какая-то проверка не проходит, большое обучение запускать бесполезно.

Обучаем собранную сеть на всех данных.

In [ ]:
torch.manual_seed(0)
model = DeepMLP(64, 10)                               # новая сеть: проверки выше её уже обучили
loss, acc = fit(model)                                # 30 эпох на всех обучающих данных
print(f"линейных слоёв: {count_linear(model)} | потеря на обучении {loss:.4f} | точность на валидации {acc:.3f}")

**Что получилось.** Точность 97.3% — потолок для этих данных, его достигают и неглубокие сети. Ценность шаблона — он **надёжно обучается** при росте глубины и ширины. Так устроены блоки трансформеров.